Downloading file about CO2 and Greenhouse Gas Emissions

In [0]:
import requests

csv_url = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"
file_path = "owid-co2-data.csv"
r = requests.get(csv_url)
with open(file_path, "wb") as f:
  f.write(r.content)
if r.status_code == 200:
  print("Downloaded f'{file_path}'")
else:
    print("Error downloading file")



In [0]:
df = spark.read.csv("file:/Workspace/Users/radek.gryciukk@softserve.academy/softserve_academy/Lab1/owid-co2-data.csv", header=True, inferSchema=True)
display(df)

## Loading the data into Delta table

In [0]:
login = "radekgryciukk"
schema = f'dbr_dev.{login}'
table = f'{schema}.co2_owid_data'

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

df.write.mode("overwrite").saveAsTable(table)

## Basic operations

In [0]:
#Selecting various types of informations
from pyspark.sql import functions as F

df = df.select("country", "year", "co2", "co2_growth_prct", "co2_growth_abs", "co2_per_capita", "co2_per_gdp", "co2_per_unit_energy", "coal_co2")
df.show(5)

In [0]:
#Annual CO2 emissions from coal in Million tonnes for choosen countries
countries = ["Poland", "Spain", "France", "Germany", "China", "India", "United States"]

df_emmision = df.filter((F.col("year") == 2024) & (F.col("country").isin(countries))) \
    .orderBy(F.col("coal_co2").desc()) \
    .limit(10) 

df_emmision.show()

In [0]:
df_filtered = df.filter(
    (F.col("country") == "Poland") & (F.col("year") >= 2022)
)
df_filtered.show()

Calling API for database

In [0]:
import requests

url = "https://countries.dev/countries"
resp = requests.get(url)
data_json = resp.json()

countries = [
    (
        c.get("alpha3Code"),
        c.get("region"),
        c.get("subregion")
    )
    for c in data_json
    if c.get("alpha3Code") is not None
]
df_regions = spark.createDataFrame(countries, ["iso_code", "region", "subregion"])
df_regions.write.mode("overwrite").saveAsTable("dbr_dev.radekgryciukk.api_regions")
df_regions.show(5)

In [0]:
df_emission = spark.table("dbr_dev.radekgryciukk.co2_owid_data")
df_regions = spark.table("dbr_dev.radekgryciukk.api_regions")

df_combined = df_emission.join(df_regions, on = "iso_code", how = "inner")

In [0]:
df_combined.show(5)

In [0]:
from pyspark.sql import functions as F

df_clean = df_combined.filter(
    (F.col("year") == 2022) &
    (F.col("coal_co2").isNotNull()) &
    (F.col("subregion").isNotNull())
).select("country", "coal_co2", "subregion", "region", "year")

df_regional_sum = df_clean.groupBy("region", "subregion") \
    .agg(F.round(F.sum("coal_co2"), 2).alias("total_coal_co2_2022")) \
    .orderBy(F.col("total_coal_co2_2022").desc())

df_regional_sum.write.format("delta").mode("overwrite").saveAsTable("dbr_dev.radekgryciukk.coal_co2_emission_regional")
display(df_regional_sum)